# Smoke test: jupyterlab_git_core

Tests the `jupyterlab-git-core` package: the Python git engine (`jupyterlab_git_core.git.Git`) and the
bundled JupyterLab front-end files that the `jupyterlab-git` extension builds on. Nothing here needs the
`git` command or a running server.

In [ ]:
# Example 1: module version, distribution metadata and the bundled JupyterLab extension files
import importlib.metadata
import json
from pathlib import Path

import jupyterlab_git_core

dist_version = importlib.metadata.version("jupyterlab-git-core")
print(f"jupyterlab_git_core.__version__ = {jupyterlab_git_core.__version__}; distribution = {dist_version}")
assert jupyterlab_git_core.__version__ == dist_version, (jupyterlab_git_core.__version__, dist_version)

paths = jupyterlab_git_core._jupyter_labextension_paths()
print(f"labextension paths: {paths}")
assert len(paths) == 1 and paths[0]["dest"] == "@jupyterlab/git", paths

ext_dir = Path(jupyterlab_git_core.__file__).parent / paths[0]["src"]
package_json = json.loads((ext_dir / "package.json").read_text())
print(f"bundled front end: {package_json['name']} {package_json['version']}")
assert package_json["name"] == paths[0]["dest"], package_json["name"]

In [ ]:
# Example 2: import the Git handler class from jupyterlab_git_core.git
from jupyterlab_git_core import git as git_module

print(f"Module: {git_module.__name__}")
print(f"File: {git_module.__file__}")

# The primary class is Git, which provides async methods to run git operations.
assert hasattr(git_module, 'Git'), "expected Git class in jupyterlab_git_core.git"
Git = git_module.Git
print(f"Git class: {Git}")
print(f"Git is class: {isinstance(Git, type)}")

In [ ]:
# Example 3: instantiate Git and inspect its methods
from jupyterlab_git_core.git import Git

# Git is typically constructed with a config object; try a no-arg or simple form.
try:
    g = Git()
    print(f"Constructed Git instance: {g}")
except TypeError as e:
    print(f"Git() needs args: {e}")
    g = None

# Enumerate callable methods on the Git class itself.
methods = [m for m in dir(Git) if not m.startswith('_') and callable(getattr(Git, m, None))]
print(f"\nGit methods ({len(methods)}):")
for m in methods[:25]:
    print(f"  {m}")

assert len(methods) > 5, "expected Git to expose multiple methods"

In [ ]:
# Example 4: the Git engine's operations are coroutines, so the server can await them
import inspect

from jupyterlab_git_core.git import Git

operations = ["status", "log", "branch", "diff", "init", "clone", "add", "commit", "show", "config"]
for name in operations:
    method = getattr(Git, name, None)
    assert method is not None, f"Git has no {name}()"
    assert inspect.iscoroutinefunction(method), f"Git.{name} is not a coroutine function"
    print(f"  async {name}{inspect.signature(method)}")

In [ ]:
# Example 5: .gitignore management and the input helpers work without the git command
import asyncio
import concurrent.futures
import tempfile
from pathlib import Path

from jupyterlab_git_core.git import Git, GitParameterError, reject_option_injection, strip_and_split


def run(coro):
    # The notebook kernel already runs an event loop in this thread, so run the coroutine on a worker thread.
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coro).result()


git = Git()
with tempfile.TemporaryDirectory() as repo:
    gitignore = Path(repo) / ".gitignore"
    assert run(git.ensure_gitignore(repo)) == {"code": 0}
    assert gitignore.exists() and gitignore.read_text() == ""

    assert run(git.ignore(repo, "build/")) == {"code": 0}
    # use_extension=True ignores every file with the same extension instead of the one file
    assert run(git.update_gitignore(repo, file_path="data/table.csv", use_extension=True)) == {"code": 0}
    print(gitignore.read_text())
    assert gitignore.read_text().splitlines() == ["build/", "**/*.csv"], gitignore.read_text()

    assert run(git.write_gitignore(repo, "*.pyc")) == {"code": 0}
    assert gitignore.read_text() == "*.pyc\n"
    assert git.read_file(str(gitignore)) == {"code": 0, "content": "*.pyc\n"}
    assert git.read_file(str(Path(repo) / "missing.txt"))["code"] == -1

# Helpers for parsing `git ... -z` output and for refusing refs that git would read as options
assert strip_and_split("a.py\x00b.py\x00") == ["a.py", "b.py"]
reject_option_injection("main", "branch")
try:
    reject_option_injection("--upload-pack=touch", "branch")
except GitParameterError as e:
    print(f"rejected option-like ref: {e}")
else:
    raise AssertionError("a ref starting with '-' was accepted")